---
# Consistencia Semántica y Etiquetado de Tópicos con un LLM
## Python + Hugging Face
---

En la unidad anterior (**5a**) estimamos tres modelos LDA sobre el corpus de críticas de películas, con $k=12$, $k=20$ y $k=25$ tópicos, y exportamos las 15 palabras más representativas de cada tópico a tres archivos CSV.

En este notebook usaremos un **LLM de código abierto** (vía Hugging Face `transformers`) para dos tareas complementarias a lo que ya hicimos en R:

1. **Etiquetar** cada tópico con un nombre corto y descriptivo, a partir de sus palabras clave.
2. **Evaluar la consistencia semántica** (*topic coherence*) de cada tópico: ¿forman estas palabras un tema reconocible, o son una mezcla poco relacionada?

Esto nos da una segunda fuente de evidencia — cualitativa y semántica — para comparar los tres valores de $k$, complementando las métricas estadísticas ya vistas (perplexity, elbow method).

> 🖥️ **Antes de ejecutar: revisa tu hardware**
>
> Este notebook detecta automáticamente el hardware disponible y usa la configuración más rápida para cada caso — no necesitas cambiar nada en el código.
>
> - **Google Colab con GPU activada**, o **Mac con Apple Silicon (M1/M2/M3/M4)**: debería tardar solo un par de minutos en total.
> - **Sin GPU** (por ejemplo, Mac con chip Intel, o un PC sin GPU NVIDIA): el notebook lo detecta e imprime una advertencia (`⚠️ No se detectó GPU...`) antes de cargar el modelo, y luego corre en CPU — esto puede tomar **decenas de minutos**. Si ves esa advertencia y la celda de evaluación es muy lenta, cambia a Google Colab y activa GPU en `Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU (T4)`.
> - Si el notebook corre lento **sin** mostrar esa advertencia, algo no está funcionando como se espera — avísale a tu profesor antes de esperar a que termine.

## 🎯 Objetivos de aprendizaje

- Cargar un modelo instruccional open-source desde Hugging Face y ejecutarlo localmente o en Colab.
- Diseñar un *prompt* que le pida al LLM una salida en un formato específico y parseable.
- Aplicar el LLM sistemáticamente a los tópicos de los tres modelos ($k=12,20,25$).
- Comparar la coherencia semántica promedio entre valores de $k$ como evidencia adicional para elegir el número de tópicos.

> 📝 **Nota:** El término **parseable** (o *parsable*) se refiere a cualquier texto, dato o código que posee una **estructura clara y predecible**, permitiendo que otro programa pueda analizarlo automáticamente, descomponerlo y extraer información sin intervención humana.


## 🤖 ¿Qué modelo usamos?

Usaremos **`Qwen/Qwen2.5-3B-Instruct`**.

Es una alternativa a modelos como `meta-llama/Meta-Llama-3-8B-Instruct` con ventajas prácticas para este notebook:

- **No requiere acceso restringido ("gated")**: los pesos de Llama-3 en Hugging Face exigen aceptar una licencia y solicitar acceso (con aprobación no inmediata) antes de poder descargarlos, lo que agrega fricción a un ejercicio de curso. Qwen2.5 se descarga directamente.
- **Tamaño acotado (3B)**: para una tarea estructurada y acotada como esta (leer 15 palabras, producir una etiqueta y un puntaje), no se necesita un modelo de 7-8B parámetros. Un modelo más pequeño se descarga y carga notablemente más rápido, lo cual importa especialmente en Colab, donde la caché no persiste entre sesiones (ver nota en la siguiente celda).

> 📌 El nombre del modelo está aislado en una sola variable (`MODEL_NAME`). Si prefieres más calidad a cambio de una carga más lenta, puedes cambiarla por `"Qwen/Qwen2.5-7B-Instruct"` o, si ya tienes acceso aprobado y un token de Hugging Face configurado, `"meta-llama/Meta-Llama-3-8B-Instruct"` — el resto del notebook funciona igual.

> ⚠️ **Sobre el hardware:** ver la nota al inicio del notebook y la explicación en la sección "Cargar el modelo" — el código detecta automáticamente si usar CUDA, Apple Silicon (MPS) o CPU.

---
## 📦 Instalación de las librerías necesarias

Antes de trabajar con un **Large Language Model (LLM)**, es necesario instalar algunas bibliotecas de Python que proporcionan las herramientas para descargar, ejecutar y analizar modelos de lenguaje.

La siguiente instrucción instala automáticamente los paquetes que utilizaremos en este notebook:

```python
!pip install -q transformers accelerate torch pandas matplotlib
```

---

### 📚 ¿Para qué sirve cada biblioteca?

- **`transformers`**  
  Proporciona acceso a miles de modelos de lenguaje preentrenados disponibles en **Hugging Face**, incluyendo modelos generativos, clasificadores y modelos de *embeddings*.

- **`accelerate`**  
  Optimiza la ejecución de modelos en CPU o GPU, gestionando automáticamente el hardware disponible y facilitando el uso eficiente de los recursos computacionales.

- **`torch`**  
  Es la biblioteca de *Deep Learning* sobre la cual están implementados la mayoría de los modelos de Hugging Face. Se encarga de las operaciones matemáticas y del entrenamiento e inferencia de redes neuronales.

- **`pandas`**  
  Permite cargar, manipular y analizar tablas de datos. En este notebook se utilizará para leer el archivo CSV que contiene los tópicos generados por LDA.

- **`matplotlib`**  
  Biblioteca para crear gráficos y visualizar los resultados obtenidos durante el análisis.

---

> 📌 **Nota:** La opción `-q` (*quiet*) reduce la cantidad de mensajes mostrados durante la instalación, haciendo que la salida del notebook sea más limpia y fácil de leer.

In [4]:
!pip install -q transformers accelerate bitsandbytes torch pandas matplotlib

In [5]:
import re

import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

---
### 🔤 Expresiones Regulares (`re`)

El módulo **`re`** (*regular expressions*) proporciona herramientas para reconocer patrones de texto dentro de cadenas de caracteres (*strings*). Se utiliza comúnmente para:

- Buscar palabras o patrones específicos.
- Extraer fragmentos de texto.
- Reemplazar texto automáticamente.
- Validar si una cadena sigue un formato determinado.

> 📌 **En este notebook**, utilizaremos `re` para extraer automáticamente información de las respuestas generadas por el LLM y convertirlas en un formato que Python pueda procesar.

## 🧠 Cargar el modelo

Detectamos automáticamente el tipo de GPU disponible y cargamos el modelo de la forma más adecuada para cada caso: CUDA (con cuantización en 4 bits), Apple Silicon (MPS, sin cuantizar) o CPU como último recurso.

> ⚠️ **`bitsandbytes` (cuantización en 4 bits) solo funciona con GPUs NVIDIA (CUDA)** — por ejemplo, la GPU T4 gratuita de Google Colab. **No funciona en Mac (Apple Silicon ni Intel)**, y si el código intenta usarlo de todas formas, `device_map="auto"` puede terminar ejecutando el modelo silenciosamente en **CPU**, lo cual es extremadamente lento para un modelo de 3B (decenas de minutos para las 57 llamadas de las secciones siguientes) — sin ningún error o advertencia visible que lo indique.
>
> Por eso esta celda revisa explícitamente qué hardware hay disponible:
>
> - **CUDA disponible** (Colab, PC con GPU NVIDIA): cuantizamos en 4 bits con `bitsandbytes`.
> - **MPS disponible** (Mac con chip Apple Silicon: M1, M2, M3, M4): cargamos el modelo en `float16` directamente sobre la GPU integrada de Apple, sin `bitsandbytes`. Un modelo de 3B en `float16` ocupa ~6 GB, lo que cabe cómodamente en la memoria unificada de un Mac moderno.
> - **Ninguna de las anteriores**: el modelo corre en CPU. Si ves este mensaje y la celda de evaluación demora demasiado, considera ejecutar el notebook en Google Colab.

> ⏳ **¿Por qué la carga del modelo puede demorar?** La primera vez (o en cada sesión nueva de Colab, donde la caché no persiste), Hugging Face debe descargar los pesos del modelo — con `Qwen/Qwen2.5-3B-Instruct` esto son unos ~6 GB, notablemente menos que los ~15 GB de una variante de 7B. Si prefieres más calidad a cambio de una descarga más pesada, cambia `MODEL_NAME` por `"Qwen/Qwen2.5-7B-Instruct"` o `"meta-llama/Meta-Llama-3-8B-Instruct"` (esta última requiere solicitar acceso en Hugging Face, ver nota más arriba).

In [6]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

if torch.cuda.is_available():
    # GPU con CUDA (por ejemplo, en Google Colab): bitsandbytes permite
    # cuantizar el modelo a 4 bits para reducir el uso de memoria.
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
    )
elif torch.backends.mps.is_available():
    # Mac con chip Apple Silicon: bitsandbytes NO soporta el backend MPS de
    # Apple, así que cargamos el modelo sin cuantizar (en float16) directamente
    # en la GPU integrada. Un modelo de 3B en float16 ocupa ~6 GB, lo que cabe
    # cómodamente en la memoria unificada de un Mac moderno.
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
    ).to("mps")
else:
    # Sin GPU disponible (ni CUDA ni MPS): el modelo corre en CPU, lo cual es
    # considerablemente más lento para un modelo de 3B. Si esta celda demora
    # demasiado, considera ejecutar el notebook en Google Colab (GPU T4 gratuita).
    print("\u26a0\ufe0f No se detectó GPU (ni CUDA ni MPS). El modelo correrá en CPU y será mucho más lento.")
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

Loading checkpoint shards: 100%|██████████| 2/2 [00:12<00:00,  6.46s/it]


> ⚡ **Nota de rendimiento:** por defecto, `generate()` puede no detenerse apenas el modelo termina su respuesta y seguir generando hasta `max_new_tokens`, lo que hace que cada llamada tarde mucho más de lo necesario — con 57 tópicos en total (12+20+25), esa diferencia se acumula rápido.
>
> Para evitarlo, le indicamos explícitamente el token de fin de turno de Qwen (`<|im_end|>`) además del `eos_token_id` por defecto, y bajamos el límite a 80 tokens — de sobra para el formato de 3 líneas que pedimos en el prompt.

In [ ]:
# Qwen2.5-Instruct marca el fin de un turno con <|im_end|>, además de su eos_token_id
# por defecto. Sin este token explícito, generate() puede no detenerse ahí y seguir
# generando texto de relleno hasta max_new_tokens en cada una de las 57 llamadas.
im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
stop_token_ids = [tokenizer.eos_token_id]
if im_end_id is not None and im_end_id != tokenizer.unk_token_id:
    stop_token_ids.append(im_end_id)

In [7]:
def query_llm(prompt, max_new_tokens=80):
    """Envía un prompt al modelo usando su plantilla de chat y devuelve solo el texto generado."""
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    output = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=stop_token_ids,
        pad_token_id=tokenizer.pad_token_id,
    )

    respuesta = tokenizer.decode(output[0][input_ids.shape[-1]:], skip_special_tokens=True)
    return respuesta.strip()

## 📂 Cargar los tópicos exportados desde R (Unidad 5a)

Los tres archivos deben estar en el mismo directorio de trabajo que este notebook (o sube los archivos a Colab antes de ejecutar esta celda).

In [8]:
topterms_12 = pd.read_csv("lda_topterms_k12.csv")
topterms_20 = pd.read_csv("lda_topterms_k20.csv")
topterms_25 = pd.read_csv("lda_topterms_k25.csv")

topterms_12.head()

,topic,term,beta,k
0,1,action,0.022739,12
1,1,jackie,0.011834,12
2,1,plot,0.006915,12
3,1,crime,0.006590,12
4,1,cop,0.006266,12


## ✂️ Palabras clave por tópico

Una función simple para obtener, para un modelo y un tópico dado, sus $n$ palabras con mayor probabilidad ($\beta$) — ya vienen ordenadas desde R, pero lo hacemos explícito aquí.

In [9]:
def get_topic_words(df, topic_id, n=15):
    palabras = (
        df[df["topic"] == topic_id]
        .sort_values("beta", ascending=False)
        .head(n)["term"]
        .tolist()
    )
    return palabras

get_topic_words(topterms_20, topic_id=1)

['action',
 'jackie',
 'chan',
 'van',
 'fight',
 'plot',
 'hong',
 'kong',
 'bond',
 'chinese',
 'martial',
 'arts',
 'films',
 'damme',
 'blade']

## ✍️ Diseño del prompt

Le pedimos al modelo una salida en un **formato fijo y parseable** (tres líneas con etiquetas conocidas), en lugar de una respuesta libre — esto hace posible extraer los resultados automáticamente con una expresión regular, sin depender de que el modelo "converse" de forma predecible.

In [10]:
PROMPT_TEMPLATE = """Estas son las {n} palabras más representativas de un tópico obtenido mediante \
Latent Dirichlet Allocation (LDA), aplicado a un corpus de críticas de películas:

{words}

Responde EXACTAMENTE en este formato, sin texto adicional ni explicaciones fuera de él:

Etiqueta: <2 a 4 palabras que describan el tema común>
Coherencia (1-5): <número entero, donde 1 = las palabras no tienen relación temática entre sí y 5 = forman un tema claro y coherente>
Justificación: <una frase breve explicando tu evaluación de coherencia>
"""


def build_prompt(words):
    return PROMPT_TEMPLATE.format(n=len(words), words=", ".join(words))


print(build_prompt(get_topic_words(topterms_20, topic_id=1)))

Estas son las 15 palabras más representativas de un tópico obtenido mediante Latent Dirichlet Allocation (LDA), aplicado a un corpus de críticas de películas:

action, jackie, chan, van, fight, plot, hong, kong, bond, chinese, martial, arts, films, damme, blade

Responde EXACTAMENTE en este formato, sin texto adicional ni explicaciones fuera de él:

Etiqueta: <2 a 4 palabras que describan el tema común>
Coherencia (1-5): <número entero, donde 1 = las palabras no tienen relación temática entre sí y 5 = forman un tema claro y coherente>
Justificación: <una frase breve explicando tu evaluación de coherencia>



## 🔍 Extraer la etiqueta, el puntaje y la justificación de la respuesta

In [11]:
def parse_response(response):
    # Requerimos un ":" antes del dígito, no solo "cualquier no-dígito", porque
    # el propio texto del prompt ("Coherencia (1-5): ...") contiene un dígito
    # dentro del paréntesis — [^:]* deja pasar ese "(1-5)" completo sin
    # detenerse ahí, ya que no contiene ":", y solo captura el dígito real que
    # viene después del ":". También toleramos mayúsculas/minúsculas y posible
    # **negrita** de markdown alrededor de la etiqueta o el número.
    etiqueta = re.search(r"[Ee]tiqueta:\s*\**\s*(.+)", response)
    coherencia = re.search(r"[Cc]oherencia[^:]*:\s*\**\s*(\d)", response)
    justificacion = re.search(r"[Jj]ustificaci[oó]n:\s*\**\s*(.+)", response)

    return {
        "etiqueta": etiqueta.group(1).strip() if etiqueta else None,
        "coherencia": int(coherencia.group(1)) if coherencia else None,
        "justificacion": justificacion.group(1).strip() if justificacion else None,
    }

## 🔁 Aplicar el LLM a todos los tópicos de un modelo

Recorremos cada tópico del modelo, construimos su prompt, consultamos al LLM y guardamos el resultado parseado en un `DataFrame`.

> ⏱️ Esta celda puede tardar varios minutos por modelo, ya que genera una respuesta por cada tópico (12, 20 o 25 llamadas al modelo).

In [12]:
def evaluate_model_topics(df, k, n_words=15):
    resultados = []
    for topic_id in sorted(df["topic"].unique()):
        palabras = get_topic_words(df, topic_id, n=n_words)
        prompt = build_prompt(palabras)
        respuesta = query_llm(prompt)
        parsed = parse_response(respuesta)
        parsed.update({"k": k, "topic": topic_id, "palabras": ", ".join(palabras)})
        resultados.append(parsed)

    return pd.DataFrame(resultados)

In [13]:
resultados_12 = evaluate_model_topics(topterms_12, k=12)
resultados_20 = evaluate_model_topics(topterms_20, k=20)
resultados_25 = evaluate_model_topics(topterms_25, k=25)

resultados_todos = pd.concat([resultados_12, resultados_20, resultados_25], ignore_index=True)
resultados_todos

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The followi

KeyboardInterrupt: 

## 📊 Comparar la coherencia semántica promedio por valor de $k$

Promediamos el puntaje de coherencia asignado por el LLM dentro de cada modelo, para tener un resumen comparable con las métricas estadísticas (perplexity, método del codo) obtenidas en R.

In [ ]:
coherencia_promedio = resultados_todos.groupby("k")["coherencia"].mean()
coherencia_promedio

In [ ]:
plt.figure(figsize=(6, 4))
coherencia_promedio.plot(kind="bar", color=["#175895", "#175895", "#175895"])
plt.ylabel("Coherencia promedio (1-5, según el LLM)")
plt.xlabel("Número de tópicos (k)")
plt.title("Coherencia semántica promedio por valor de k")
plt.ylim(0, 5)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### ⚠️ Una limitación real: los puntajes se concentran en 5

Si observas la distribución completa de `coherencia` (no solo el promedio), vas a notar algo importante: la gran mayoría de los tópicos reciben un puntaje de **5**, y muy pocos reciben 4, 3 o menos — independientemente del valor de $k$.

Esto **no es un error de parseo** (ya verificamos que el formato se extrae correctamente); es una limitación real de usar un modelo de 3B para autoevaluarse con una escala numérica: tiende a ser **poco discriminante** y a concentrar sus respuestas cerca del extremo superior de la escala, en lugar de repartirlas según la calidad real de cada tópico.

¿Por qué importa esto? Porque si casi todos los tópicos obtienen 5, el **promedio por valor de $k$** pierde buena parte de su capacidad para diferenciar entre modelos — la comparación entre $k=12$, $k=20$ y $k=25$ termina siendo mucho menos informativa de lo que sería con puntajes mejor distribuidos.

> 📌 **Idea clave:** esta es una limitación genuina del enfoque LLM, no un defecto del ejercicio — y es exactamente el tipo de comparación crítica entre métodos tradicionales y LLM que buscamos hacer en este curso. Las métricas estadísticas de la Unidad 5a (perplexity, el método del codo) no tienen este problema de "techo": producen valores continuos que sí distinguen claramente entre distintos $k$. Un modelo más grande (7B o superior), o un *prompt* que pida una justificación *antes* del puntaje (para forzar un razonamiento más cuidadoso), probablemente reduciría este efecto — quedan como posibles extensiones de este ejercicio.

## 🏷️ Revisar las etiquetas generadas

Una forma rápida de inspeccionar cualitativamente los resultados: ver las etiquetas y palabras clave de un modelo en particular.

In [ ]:
resultados_20[["topic", "etiqueta", "coherencia", "palabras"]]

## ✅ Cierre

Este notebook agrega una evidencia **cualitativa y semántica** a la selección de $k$, complementando las métricas puramente estadísticas de la Unidad 5a:

- **R / Quanteda + topicmodels**: produce los tópicos y las métricas de ajuste (perplexity) de forma reproducible y computacionalmente eficiente.
- **Python + LLM**: interpreta el *contenido* de esos tópicos — algo que ninguna métrica estadística puede hacer por sí sola — y propone etiquetas listas para presentar a una audiencia no técnica.

> 📌 **Idea clave:** ni el enfoque estadístico ni el LLM son suficientes por separado para elegir y comunicar un buen modelo de tópicos — la combinación de ambos es lo que da una respuesta completa.